# 02 — Coleta de internações hospitalares (DATASUS SIH/SUS)

**Objetivo:** obter, para os municípios de SP, o número de internações de
idosos (60+) por capítulos da CID-10 associados à hipótese do estudo, ano a
ano, de Jan/2022 a Jul/2026.

## ⚠️ Correção importante: "por local de residência", não "por local de internação"

A primeira versão desta coleta usava os arquivos do TabNet **por local de
internação** — que contam a internação no município onde fica o *hospital*,
não onde o *paciente mora*. Isso invalidava o cruzamento geográfico do
estudo: municípios-polo em saúde apareciam com taxas enormes por atenderem a
região inteira, e municípios sem hospital apareciam com zero.

Os arquivos atuais são **por local de residência**, que é o correto para a
hipótese. O efeito da troca, medido nos dados:

| | por local de internação | por local de residência |
|---|---|---|
| Municípios com ao menos 1 internação | ~313 de 645 | **645 de 645** |
| Total de internações no estado | ~487 mil | ~488 mil (praticamente igual) |
| Pariquera-Açu | 2.088 | 300 (estava **7x** inflado) |
| Poá e Peruíbe | 0 | 797 e 781 |

O total do estado quase não muda (é a mesma população de internações), mas a
**distribuição entre municípios muda completamente** — que é exatamente o que
o estudo mede. Vale reportar essa correção na Metodologia do artigo.

## Os três arquivos

Um por capítulo da CID-10 (é o nível de filtro que o TabNet oferece — não dá
pra pedir só W00-W19 ou só S72 direto na interface). Seleção usada: Linha =
`Município`, Coluna = `Ano processamento`, Conteúdo = `Internações`, faixas
etárias 60-69, 70-79 e 80+.

| Arquivo | Capítulo CID-10 | Total no período |
|---|---|---|
| `sih_ano_lesoes_sp.csv` | XIX — Lesões e causas externas | 350.018 |
| `sih_ano_sintomas_sp.csv` | XVIII — Sintomas e sinais mal definidos | 108.899 |
| `sih_ano_tmentais_sp.csv` | V — Transtornos mentais e comportamentais | 29.100 |

⚠️ **Limitação a declarar no artigo:** cada capítulo é mais largo do que o
subgrupo de interesse original. O capítulo XVIII em especial é usado na
literatura de saúde pública como proxy de diagnóstico tardio/impreciso — o que
reforça a hipótese em vez de enfraquecê-la. Já o capítulo V é o mais largo dos
três (inclui transtornos por uso de substâncias, esquizofrenia etc.) — vale
tratar como complementar/exploratório, não como pilar central do argumento.

⚠️ **2026 é parcial** (Jan a Jul). Nos gráficos de série temporal ele aparece
marcado como tal, e fica **fora** de qualquer ajuste de tendência — senão a
reta cai artificialmente no último ponto.

⚠️ **"Ano processamento", não "ano de atendimento":** é o ano em que a AIH foi
processada pelo sistema, que pode ser 1-2 meses depois da internação. É o
padrão do DATASUS e o que mantém consistência com o filtro de período —
declarar no artigo.

**Saída:** `data/processed/internacoes_sp.csv`, uma linha por (município, ano,
causa).


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd
import io


## 2.1 Parser do formato TabNet (testado com os arquivos reais)

O TabNet exporta em **Latin-1** (não UTF-8), separado por `;`, com linhas de
metadados antes da tabela e notas de rodapé depois — por isso não dá pra usar
`pd.read_csv` direto, é preciso recortar só o miolo primeiro.

Validado: a soma de cada arquivo bate exatamente com o "Total" do rodapé
(350.018 / 108.899 / 29.100), e os nomes de município cruzam 100% com o Censo
(o único ajuste de grafia, "São Luís" vs. "São Luiz" do Paraitinga, está em
`config.ALIASES_MUNICIPIO`).


In [ ]:
def parse_tabnet_sih(path, causa_label):
    """Lê um export do TabNet SIH (um capítulo, todos os municípios de SP, uma
    coluna por ano) e devolve um DataFrame longo:
    municipio_norm, ano, causa, internacoes."""
    with open(path, encoding="latin1") as f:
        lines = f.readlines()

    # a tabela começa no cabeçalho "Município" e termina antes da linha
    # "Total" (soma geral), seguida das notas de rodapé
    header_idx = next(i for i, l in enumerate(lines) if l.startswith('"Munic'))
    end_idx = next(i for i, l in enumerate(lines) if l.startswith('"Total"'))

    df = pd.read_csv(io.StringIO("".join(lines[header_idx:end_idx])), sep=";", quotechar='"')
    df = df.rename(columns={df.columns[0]: "municipio_raw"})

    # a primeira coluna vem como "350010 ADAMANTINA" -- descarta o código e
    # normaliza o nome, que é a chave de cruzamento entre as bases
    nome = df["municipio_raw"].str.replace(r"^\d{6}\s+", "", regex=True)
    df["municipio_norm"] = nome.apply(config.normalizar_municipio)

    anos = [c for c in df.columns if c.isdigit()]  # ignora a coluna "Total"
    for col in anos:
        df[col] = df[col].replace("-", "0").astype(int)  # "-" no TabNet = zero

    longo = df.melt(id_vars="municipio_norm", value_vars=anos,
                    var_name="ano", value_name="internacoes")
    longo["ano"] = longo["ano"].astype(int)
    longo["causa"] = causa_label
    return longo[["municipio_norm", "ano", "causa", "internacoes"]]


In [ ]:
arquivos = {
    "lesoes_causas_externas": "sih_ano_lesoes_sp.csv",
    "sintomas_sinais_maldefinidos": "sih_ano_sintomas_sp.csv",
    "transtornos_mentais": "sih_ano_tmentais_sp.csv",
}

internacoes = pd.concat(
    [parse_tabnet_sih(config.DATA_EXTERNAL / arq, causa) for causa, arq in arquivos.items()],
    ignore_index=True,
)

print(internacoes.shape)
print(f"{internacoes['municipio_norm'].nunique()} municípios distintos")
print()
print(internacoes.pivot_table(index="causa", columns="ano", values="internacoes",
                              aggfunc="sum", margins=True, margins_name="Total"))


**Conferência:** a coluna `Total` acima deve dar 350.018 (lesões), 108.899
(sintomas) e 29.100 (transtornos mentais) — os mesmos valores do rodapé de
cada CSV.

Note que transtornos mentais aparece em 584 municípios, não 645: nesse
capítulo, 61 municípios não tiveram nenhuma internação de idoso no período. O
notebook 03 trata isso preenchendo com 0.

A queda aparente em 2026 é só o ano estar incompleto (até julho).


## 2.2 Salvar resultado consolidado


In [ ]:
internacoes.to_csv(config.DATA_PROCESSED / "internacoes_sp.csv", index=False)
print("Salvo em", config.DATA_PROCESSED / "internacoes_sp.csv")


## 2.3 (Opcional / mais adiante) Microdados via `pysus`

Se um dia quisermos refinar para o nível de subcategoria (só W00-W19, só S72
etc., em vez do capítulo inteiro), o caminho é baixar os microdados de AIH
direto do SIH e classificar `DIAG_PRINC` nós mesmas — `config.py` já tem
`CAUSAS_CID10` e `classificar_causa()` prontos para isso. **Não é necessário
agora** (a seção 2.1 resolve a coleta principal).

⚠️ **Testado no ambiente de desenvolvimento (sem internet externa):**
`pip install pysus` funciona, mas a chamada que baixa os dados
(`pysus.sih(...)`) falha com `ProxyError 403 Forbidden` — o proxy não libera
os servidores do DATASUS. **Esta célula só funciona rodando localmente.**
Note também que a API do `pysus` mudou: a função antiga
`pysus.online_data.SIH.download` não existe mais nas versões recentes.


In [ ]:
try:
    import pysus
    PYSUS_OK = True
except ImportError as e:
    print("pysus não instalado (rode: pip install pysus). Não é necessário para o caminho principal:", e)
    PYSUS_OK = False

# Exemplo, para rodar localmente (fora deste ambiente, que bloqueia o DATASUS):
# df_mes = pysus.sih(config.UF_SIGLA, 2022, 1, as_dataframe=True)
# df_mes["causa"] = df_mes["DIAG_PRINC"].apply(config.classificar_causa)
